# Import Cell

In [1]:
import torch
import numpy as np
import time
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Difference Between GPU and CPU usage

The format specifier :.4f determines how a decimal number (floating-point number) is formatted and displayed in programming languages like Python.
It means the number will be displayed as a fixed-point decimal rounded to exactly 4 decimal places.
Here is how the syntax breaks down:
- : Signals the start of the formatting rules.
- .4 Specifies the precision, forcing exactly 4 digits after the decimal point.
- f Represents "fixed-point" notation, which displays the number as a standard decimal rather than in scientific notation.

We can also use %%time as a header to record the time taken for execution.
What happens realtime is this, cpu executes but there are some waiting times in between blocks of execution. 
```CPU times``` shows cpu calculations only while ```Wall times``` include those wait times as well.

In [2]:
%%time
# we are using time module of python
start_time=time.time() # It will store the time from epoch in ms on execution at this line

# Matrix Operation Here
zeros=torch.zeros(1,1)
end_time=time.time()

delta_time=end_time-start_time
print(f'{delta_time:.4f}')


0.0002
CPU times: user 157 μs, sys: 29 μs, total: 186 μs
Wall time: 299 μs


```0.0000```
- ```CPU times```: user 136 μs, sys: 5 μs, total: 141 μs; Calculates the cpu usage
- ```Wall time```: 134 μs; Calculates the total time from execution and side-effect appearing

## Let's Test which one is faster on gpu, torch or numpy

In [3]:
torch_rand1=torch.rand((10000,10000),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((10000,10000),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(10000,10000).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(10000,10000).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00019813 fucking screaming fast!
# For float64 0.00018287 SAME SPEED!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 4.21621251 screaming slow!
# For float64 10.94090414; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09828210 okay, but definitely not doing the job!
# For float64 0.17952561; double as expected
# Shape is simple, so cpu is better at it!

0.58808661
11.10464859
0.20761204


## Let's Test a shape of (100,100,100,100) so 4D, let's see who wins

In [4]:
torch_rand1=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(100,100,100,100).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(100,100,100,100).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00027418 slower than 2D!
# For float64 0.00026464 even Faster!!!!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.16696119 great performance!
# For float64 0.35045409; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09236670 okay, but definitely not doing the job!
# For float64 0.17640281; double as expected
# Shape is simple, so cpu is better at it!

0.07797217
0.35858369
0.19244719


# Billions of But Simple Computation
When we will have millions or billions of parameters for our model, we will be doing very simple computations.
Above example demonstrates real time how efficient my rtx 3060 ti 8GB gpu is compared to 10700k.

## Let's Now Work on Some torch functions

In [5]:
# Define a probability tensor
probabilities=torch.tensor([0.1,0.9])
# 10% or .1->0.90% or .9->1; probabilities[0],probabilities[1]=0.1,0.9 
# Each probability points to the index of the of the probability in the tensor and they have to add up to 1
# Draw 5 samples from multinomial distribution
samples=torch.multinomial(probabilities,num_samples=10,replacement=True)
# Multinomial takes in a probability tensor object of 1 by 2 as its first argument and creates a 1D tensor consisting of 0s and 1s.
# Multinomials shows the indexes in probability order or bias, remember that wighted walk by mewstro?
# According to our script the index 0 has 10% chance to exist and index 1 has 90% chance to exist, and torch.multinomial returns a random 1D tensor with the indexes as elements(booleans for something else?).
print(samples)

tensor([1, 1, 1, 1, 1, 1, 1, 0, 1, 1])


We are going to use this to predict what words gonna come next as biases(or random walks).

In [6]:
# torch.cat() concatenates two tensors like lists
tensor=torch.tensor([1,2,3,4])
out=torch.cat((tensor,torch.tensor([5])),dim=0)
out

tensor([1, 2, 3, 4, 5])

Above torch.cat() is same as the concatenation below.

In [7]:
a=[1,2,3,4]
b=[5]
a+b

[1, 2, 3, 4, 5]

## We Are Going To Actually Use This During Generation
1. It's going to start from zero, we are going to use our probability of distribution to pick the first character.
2. Based on the first character we are going to pick the second one and then ```concatenate``` with the first one.

For example if we have 100 characters here and remaining 100 there, we are going to concatenate them using this method. And in the end we will have all of these integers that we have predicted.

## tril Stands For Triangle Lower(Including The Diagonal)

In [8]:
# tril is gonna multiply the upper part of the diagonal with zero
out=torch.tril(torch.ones(5,5))
out=torch.tril(torch.ones(5,5))
out

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

# The Triangle Shows Time
First we predict 1 and remaining 4 is zero in the first row but slowly the ones talk with the next one and we have 3 zeros and in the last tow we just finished predicting all 5 of them.
```
[[1., 0., 0., 0., 0.],
 [1., 1., 0., 0., 0.],
 [1., 1., 1., 0., 0.],
 [1., 1., 1., 1., 0.],
 [1., 1., 1., 1., 1.]]
```
In the end we have more and more history to look back into rather than future.
## Premise:
Basically the premise of this is to make sure that we can't communicate with the answere, just like in the exam hall. Opposit case for triu;
```
[[1., 1., 1., 1., 1.],
 [0., 1., 1., 1., 1.],
 [0., 0., 1., 1., 1.],
 [0., 0., 0., 1., 1.],
 [0., 0., 0., 0., 1.]]
```
## triu Stands For Triangle Upper(Including The Diagonal)

In [9]:
# triu is gonna multiply the lower part of the diagonal with zero
out=torch.triu(torch.ones(5,5))
out

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

# Mask
When we store a boolean condition with a given tensor containing 1s or 0s, we get a mask, a tensor with trues where boolean satisfies and falses where it doesn't. a mask as such, a tensor of boolean value is insertable into the ```masked_fill()``` method of another tensor object;
```
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
```
The Trues get multiplied element-wise with the 2nd args provided as an float() or int() object and gives a tensor as;
```
tensor([[False,  arg2,   arg2,  arg2,  arg2 ],
        [False,  False,  arg2,  arg2,  arg2 ],
        [False,  False,  False, arg2,  arg2 ],
        [False,  False,  False, False, arg2 ],
        [False,  False,  False, False, False]])
```

In [10]:
mask=torch.tril(torch.ones(5,5))==0
masked=torch.zeros(5,5).masked_fill(mask,float('-inf'))
masked

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

# Exponentiate Refers to Natural Exoinential Function
Since we were just working with `masked_fill` and `float('-inf')`, exponentiation is the very next step in the **Softmax** formula where $e^{0}=0$;

* **Positive numbers grow fast:** Larger scores become much larger weights.
* **Negative numbers become tiny positive fractions:** For example, $e^{-2} \approx 0.1353$.
* **Negative infinity (`-inf`) becomes `0`:** Because $e^{-\infty} = 0$, filling masked spots with `-inf` ensures they turn into `0.0` after exponentiation and get completely ignored by the model.

In [11]:
# Let's use torch.exp().
torch.exp(masked)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

## '-inf' Becomes 0 and 0 becomes 1
```
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
```
## Transpose is More Like Swapping

In [12]:
input=torch.zeros(2,3,4)
# By default torch.zeros() takes dimension as args.
# We can use method transpose() to swap the dimension value from the input tensor.
# .transpose() takes is index th number of the dimensions and swaps them.
out=input.transpose(0,2)
# Here we just swaped 0th dimension 2 with 2th dimention 4.
input,out

(tensor([[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],
 
         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]),
 tensor([[[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]]]))

# Stack Tensors on Next Dimension
torch.stack([torch.tensor]) takes in tensor objects and house them serially as rows in the direction of the column space. It can take in tensors as opposed to torch.tensor([list(]) which takes in list objects only.
- Same as multiplying a vector $(\vec{a}_j-y_i)$ with a transpose $(\vec{a}_j^0)^T$ to get a 2d matrix.

We can stack from $1D\rightarrow 2D$ or any $(n-1)D\rightarrow nD$

## Relevance to Bach Size

In [13]:
tensor1=torch.tensor([1,2,3])
tensor2=torch.tensor([4,5,6])
tensor3=torch.tensor([7,8,9])

# Stack along next dimension
stacked=torch.stack([tensor1,tensor2,tensor3])
stacked

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

# Neural Network NN
It's very important for the model it's called the nn.linear function. nn.Linear() has weights and biases and it learns and gets better.
Let's go to torch.org torch.nn documentation. We have containers and a lot of layers.
For our example with nn.Linear(), we will start with a 3 $\vec{a}_3^0$ inputs and 3 $\vec{a}_3^L$ outputs; usually vectors of equal length $(\vec{a}^0_j\rightarrow\vec{a}_j^L)$ and compute;
$$W^L_{(3,3)}\times \vec{a}_3^0+b^L=\vec{a}_3^L$$
For nn.Sequential() has in between layers, and we will have to sync our $\vec{W}^l_{ij}$ weights with each $\vec{a}_3^{l-1},\vec{a}_3^l$ so that math can do the mathing.

In [14]:
import torch.nn as nn
sample=torch.tensor([10.,10.,10.])
# Must be floats
linear=nn.Linear(3,3,bias=False)
linear(sample)

tensor([-1.5887, -5.5918,  6.8945], grad_fn=<SqueezeBackward4>)

# Softmax Function
Softmax function isn't actually what it sounds like at all. When we shove our tensor into it we are doing exactly this to each of the tensor members; 
$$tensor[i]'=\frac{e^{tensor[i]}}{\sum_{i=1}^m e^{tensor[i]}}$$
In more accurate language the standard unit softmax function $\sigma:\mathbb{R}^K\rightarrow(0,1)^K$ is defined when $K\ge1$ by the formula
$$\sigma(z)_i=\frac{e^{z_i}}{\sum_{i=1}^m e^{z_i}}\text{ for }i=1,...,K\text{ and z}=(z_1,...,z_K)\in\mathbb{R}^K$$
This is also called normalization or an activation function as opposed to relu or max(0,1).

In [15]:
import torch.nn.functional as F
# Create a tensor
tensor1=torch.tensor([1.,2.,3.])
# Apply softmax using torch.nn.functional.softmax()
softmax_output=F.softmax(tensor1,dim=0) # Scalar?
softmax_output

tensor([0.0900, 0.2447, 0.6652])

# nn.Embedding 
It is a class in torch that represents an embedding layer which represents a vecor of discrete inputs, such as words as categorical variables, as a continuous stream of long vector of values, our $\vec{a}^0_j$ layer.
The nn.Embedding layer maps each discrete input to a dense vector representation, often referred to as an embedding vector. They capture the semantic relationships between the inputs, allowing the model to learn meaningful representation for the discrete inputs. Basically numeric representation in a simple long vector.
## The gpt Generated Code as an Example Below

In [23]:
# initialize an embedding layer
vocab_size=1000
embedding_dim=100 # Number of elements per stack
embedding=nn.Embedding(vocab_size,embedding_dim) # The so called argument it takes in
# Create some input indices, just random
input_indices=torch.LongTensor([1,5,3,2]) # Note that it is made of int64 as indices and determines the number of stacks.
input_indices2=torch.LongTensor([1,1,1,1])
# Apply the embedding layer
embedded_output=embedding(input_indices) # It only takes in ints so int64
# These nn.blablas are functions assigned like objects where variables latter produce () s
embedded_output2=embedding(input_indices2)
embedded_output,embedded_output2
# cell output <torch.Size([4, 100])>, sampling fot large values?

(tensor([[ 4.4588e-01, -7.6048e-01, -1.0054e-01, -1.3668e+00,  1.6499e+00,
           8.8490e-01, -1.1585e+00, -4.3810e-01, -4.5875e-01, -1.7010e-01,
           1.2862e+00,  9.1169e-01, -7.9773e-02,  1.0458e+00,  2.0217e-02,
           3.9989e-01, -1.1414e-01, -3.3582e-01,  1.1120e+00, -7.7809e-01,
           2.2823e-01, -1.3679e+00,  3.9753e-01, -2.3544e-01,  1.4816e+00,
           1.4186e+00, -1.7154e+00, -8.3136e-01,  3.7509e-01, -1.2822e+00,
          -8.1045e-01,  5.6690e-01, -6.4515e-01,  9.5942e-01, -6.8247e-02,
          -5.1164e-01, -7.8144e-02, -1.0500e+00,  1.7273e-01,  3.7083e-02,
           4.1759e-01,  6.3226e-01, -9.3380e-01, -1.2180e-01, -2.6159e+00,
           1.3996e+00,  1.6468e+00,  1.3049e+00,  1.2052e+00, -2.0702e-01,
          -2.5256e-01,  2.8926e+00,  5.1045e-01,  8.0947e-01, -1.3743e+00,
          -5.9482e-01,  6.7311e-01, -8.2475e-01, -1.9307e-01,  7.4031e-01,
          -5.7989e-01, -1.0975e-01, -2.9262e-01, -1.6013e+00,  1.3126e-01,
           1.0085e-01,  9

# Gemini Generated

In [55]:
# Define an embedding layer for a vocab of 10 words, each mapped to a 3D vector
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3, padding_idx=0)
'''Creates a Raw embeding parameter table of size(10,4). embedding.detach() is a tensor'''

# Input must be a LongTensor (integers) of token indices
# Shape: (batch_size=2, sequence_length=4)
input_indices = torch.LongTensor([
    [1, 2, 4],
    [4, 3, 2],  # 0 is the padding index
    [4, 3, 0]])
'''this is -> output = [embedding[i] for i in input_indices]'''

# Forward pass: retrieves the vectors
output = embedding(input_indices)
print(output,output.view(9,3))
#output=output.view(9,3)
print(output.shape)  # Output: torch.Size([2, 4, 3])
embedding,output,output[:,-1,:],embedding.weight.detach().numpy(),embedding.weight

tensor([[[-1.0067,  0.5179, -0.4759],
         [ 0.2890, -0.1863,  1.4781],
         [ 0.0433,  0.3587,  0.0123]],

        [[ 0.0433,  0.3587,  0.0123],
         [ 0.0310, -1.1690,  0.6718],
         [ 0.2890, -0.1863,  1.4781]],

        [[ 0.0433,  0.3587,  0.0123],
         [ 0.0310, -1.1690,  0.6718],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>) tensor([[-1.0067,  0.5179, -0.4759],
        [ 0.2890, -0.1863,  1.4781],
        [ 0.0433,  0.3587,  0.0123],
        [ 0.0433,  0.3587,  0.0123],
        [ 0.0310, -1.1690,  0.6718],
        [ 0.2890, -0.1863,  1.4781],
        [ 0.0433,  0.3587,  0.0123],
        [ 0.0310, -1.1690,  0.6718],
        [ 0.0000,  0.0000,  0.0000]], grad_fn=<ViewBackward0>)
torch.Size([3, 3, 3])


(Embedding(10, 3, padding_idx=0),
 tensor([[[-1.0067,  0.5179, -0.4759],
          [ 0.2890, -0.1863,  1.4781],
          [ 0.0433,  0.3587,  0.0123]],
 
         [[ 0.0433,  0.3587,  0.0123],
          [ 0.0310, -1.1690,  0.6718],
          [ 0.2890, -0.1863,  1.4781]],
 
         [[ 0.0433,  0.3587,  0.0123],
          [ 0.0310, -1.1690,  0.6718],
          [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>),
 tensor([[ 0.0433,  0.3587,  0.0123],
         [ 0.2890, -0.1863,  1.4781],
         [ 0.0000,  0.0000,  0.0000]], grad_fn=<SelectBackward0>),
 array([[ 0.        ,  0.        ,  0.        ],
        [-1.006715  ,  0.5178767 , -0.47586265],
        [ 0.2889565 , -0.18634461,  1.4781001 ],
        [ 0.03102122, -1.1689924 ,  0.67176586],
        [ 0.04332986,  0.35868597,  0.01229863],
        [ 1.0466887 ,  0.02659186, -0.5718394 ],
        [ 0.96428835, -0.07063713,  0.15525116],
        [-0.7855    , -0.13301563,  0.617793  ],
        [-1.0514307 , -0.131424  , -0.165

In [17]:
import random
# torch.rand() doesnt allow int so use randint
a=torch.randint(10,(3,2))
# arg1 is the range of selection like random.randint() and the tuple is the shape
b=torch.randint(10,(2,3))
[a,b,(a@b),torch.matmul(a,b)]
# @ denotes the matmul, no pun intended, remember jacobian chain rule and compute per node?

[tensor([[0, 0],
         [4, 3],
         [1, 7]]),
 tensor([[9, 0, 8],
         [8, 8, 6]]),
 tensor([[ 0,  0,  0],
         [60, 24, 50],
         [65, 56, 50]]),
 tensor([[ 0,  0,  0],
         [60, 24, 50],
         [65, 56, 50]])]

In [18]:
int_64=torch.randint(4,(3,2))
# type int64
float_32=torch.rand(2,3)
# type float32
#result=torch.matmul(int_64,float_32)
# We can not multiply different datatypes.
# It will get this error, <RuntimeError: expected m1 and m2 to have the same dtype, but got: long int != float>
# We can solve this by casting float method on the int_64 
result=torch.matmul(int_64.float(),float_32)
result
# Casting .float() is not the best, it strips away a lot of prescision.

tensor([[2.4499, 0.9807, 0.5636],
        [2.7481, 1.0142, 0.1527],
        [3.0677, 1.2853, 1.0253]])

# Multilateral Perceptron
Yeah, that's what NNs are called. Now let's go back to our bigram.ipynb notebook

In [67]:
torch.tensor(1).item()

1

In [ ]:
# Example of .shape and .view;
a=torch.rand(2,3,5)
x,y,z=a.shape # It will assign a touple to our variable touple
print(a,a.view(x*y,z),a.view(x,y*z),a.view(x*y*z))
# View is used to redistribute the elements into all possible sizes, but total item numbers 30 has to be equal in each cases.

In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Model layers: Vocab of 10 words -> 4D embedding -> 10 output logits
embedding = nn.Embedding(num_embeddings=10, embedding_dim=4)
linear = nn.Linear(in_features=4, out_features=10)

# 2. Input token IDs and Target next-word IDs (batch of 2)
input_ids = torch.LongTensor()   # Shape: (2,)
target_ids = torch.LongTensor()  # True next words: ID 5 and ID 2

# 3. Forward pass to get raw logits (z)
x = embedding(input_ids)  # Shape: (2, 4)
logits = linear(x)        # Shape: (2, 10) -> raw z scores for each of the 10 words
# 4. Calculate loss directly from raw logits
loss = F.cross_entropy(logits, target_ids)
print(loss)  # Single scalar tensor, e.g., tensor(2.3412, grad_fn=<NllLossBackward0>)
x,logits

tensor(nan, grad_fn=<NllLossBackward0>)


(tensor([], size=(0, 4), grad_fn=<EmbeddingBackward0>),
 tensor([], size=(0, 10), grad_fn=<AddmmBackward0>))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# ==========================================
# 1. VOCABULARY (5 unique characters)
# ==========================================
char_to_id = {'<PAD>': 0, 'h': 1, 'e': 2, 'l': 3, 'o': 4}
id_to_char = {0: '<PAD>', 1: 'h', 2: 'e', 3: 'l', 4: 'o'}
vocab_size = len(char_to_id)  # 5
hidden_size = 3               # 3 middle-layer neurons

# ==========================================
# 2. TRAINING DATA ("hello")
# ==========================================
input_ids = torch.LongTensor((1, 2, 3, 3))   # ['h', 'e', 'l', 'l']
target_ids = torch.LongTensor((2, 3, 3, 4))  # ['e', 'l', 'l', 'o']

# ==========================================
# 3. THE MODEL (5 Inputs -> 3 Emb -> 3 RNN Memory -> 5 Outputs)
# ==========================================
class CharRNNPredictor(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=hidden_size,
            padding_idx=0
        )
        self.rnn = nn.RNN(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.output_layer = nn.Linear(
            in_features=hidden_size,
            out_features=vocab_size
        )

    def forward(self, x):
        x = x.unsqueeze(0)                              # Shape: (1, 4)
        emb = self.embedding(x)                         # Shape: (1, 4, 3)
        rnn_out, _ = self.rnn(emb)                      # Shape: (1, 4, 3) -> memory across steps
        logits = self.output_layer(rnn_out).squeeze(0)  # Shape: (4, 5)
        return emb, rnn_out.squeeze(0), logits

model = CharRNNPredictor(vocab_size=5, hidden_size=3)

# ==========================================
# 4. TRAINING LOOP (200 Epochs)
# ==========================================
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 201):
    optimizer.zero_grad()
    _, _, logits = model(input_ids)
    loss = F.cross_entropy(logits, target_ids)
    loss.backward()
    optimizer.step()

    if epoch in (1, 50, 100, 200):
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# ==========================================
# 5. INSPECT MEMORY & PREDICTIONS AFTER TRAINING
# ==========================================
with torch.no_grad():
    _, memory_states, final_logits = model(input_ids)
    probabilities = F.softmax(final_logits, dim=-1)
    predicted_ids = torch.argmax(probabilities, dim=-1)

print("\n--- Hidden Memory State at Each Letter ---")
for i in range(len(input_ids)):
    ch = id_to_char[input_ids[i].item()]
    mem = tuple(round(v, 2) for v in memory_states[i].tolist())
    print(f"Step {i} (saw '{ch}') -> RNN Memory Vector: {mem}")

print("\n--- Final Predictions ---")
for i in range(len(input_ids)):
    in_char = id_to_char[input_ids[i].item()]
    true_char = id_to_char[target_ids[i].item()]
    pred_char = id_to_char[predicted_ids[i].item()]
    probs_rounded = tuple(round(p, 2) for p in probabilities[i].tolist())
    print(f"Input: '{in_char}' | Target: '{true_char}' | Predicted: '{pred_char}' | Probs: {probs_rounded}")


In [ ]:
for epoch in range(num_epochs):             # Outer loop: passes through entire dataset
    for input_ids, target in dataloader:    # Inner loop: one batch at a time
        optimizer.zero_grad()               # 1. Clear old gradients
        logits = model(input_ids)           # 2. ONE Forward pass
        loss = F.cross_entropy(logits, target)  # 3. Grade the batch
        loss.backward()                     # 4. ONE Backward pass
        optimizer.step()                    # 5. Update weights (Embedding + Linear)
